# Building Image Generation Applications (OpenAI)

There's more to LLMs than text generation. You can also generate images from text descriptions. Images as a modality are useful across MedTech, architecture, tourism, game development, marketing, and more. In this lesson we work with today's **GPT Image** models on the OpenAI platform.

## Learning goals

By the end of this lesson you'll be able to:

- Explain what image generation is and where it's useful.
- Understand the `gpt-image` model family and how it differs from the legacy DALL·E models.
- Build an image generation application and edit images.

## What is image generation?

Image generation models create images from a text prompt. Modern models such as `gpt-image` learn the relationship between text and images during training, then iteratively turn random noise into an image that matches your prompt.

Two well-known families of image models are:

- **`gpt-image` (OpenAI)** — the current generation used in this lesson. It supports text-to-image generation and image editing (inpainting with a mask).
- **Midjourney** — a popular third-party model with its own service and Discord-based workflow.

> The older OpenAI image models — **DALL·E 2** and **DALL·E 3** — are legacy. DALL·E 3 is no longer available for new deployments, and features like `create_variation` only existed in DALL·E 2. Use the `gpt-image` models for new applications.

> **Important:** `gpt-image` models return the generated image as **base64** (`b64_json`), not as a URL. Your code decodes the base64 string to bytes and saves it — there's no image URL to download.


## Az első képgeneráló alkalmazásod elkészítése

Akkor mire van szükség egy képgeneráló alkalmazás elkészítéséhez? A következő könyvtárakra:

- **python-dotenv**, erősen ajánlott ezt a könyvtárat használni ahhoz, hogy a titkos adataidat egy *.env* fájlban, a kódtól távol tarthasd.
- **openai**, ezt a könyvtárat fogod használni az OpenAI API-val való kommunikációhoz.
- **pillow**, hogy képekkel dolgozhass Pythonban.


1. Hozz létre egy *.env* fájlt a következő tartalommal:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Gyűjtsük össze a fenti könyvtárakat egy *requirements.txt* nevű fájlba így:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Ezután hozzuk létre a virtuális környezetet, és telepítsük a könyvtárakat:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Windows rendszeren az alábbi parancsokat használhatod a virtuális környezet létrehozásához és aktiválásához:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Add hozzá a következő kódot egy *app.py* nevű fájlba:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Hozzon létre egy OpenAI objektumot (beolvassa az OPENAI_API_KEY-t a .env fájlból)
    client = openai.OpenAI()


    try:
        # Kép létrehozása a képalkotó API használatával
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Írja be ide a prompt szövegét
            size='1024x1024',
            n=1
        )
        # Állítsa be a tárolt kép könyvtárát
        image_dir = os.path.join(os.curdir, 'images')

        # Ha a könyvtár nem létezik, hozza létre
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Inicializálja a kép elérési útját (figyeljen, hogy a fájltípus png legyen)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # A gpt-image modellek a képet base64-ben (b64_json) adják vissza, nem URL-ként
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Megjeleníti a képet az alapértelmezett képfeldolgozóban
        image = Image.open(image_path)
        image.show()

    # kivételek kezelése
    except openai.BadRequestError as err:
        print(err)

    ```

Magyarázzuk meg ezt a kódot:

- Először importáljuk a szükséges könyvtárakat, beleértve az OpenAI könyvtárat, a dotenv könyvtárat, a base64 modult és a Pillow könyvtárat.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Ezután létrehozzuk az ügyfelet, amely beolvassa az API kulcsot a ``.env`` fájlból.

    ```python
    # OpenAI objektum létrehozása
    client = openai.OpenAI()
    ```

- Következő lépésként létrehozzuk a képet:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Írd be ide a kért szöveget
        size='1024x1024',
        n=1
    )
    ```

    A `gpt-image` modellek az képet **base64** stringként adják vissza a `data[0].b64_json` mezőben. Ezt dekódoljuk bájtokká, majd fájlba írjuk — nincs letölthető URL.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Végül megnyitjuk a képet, és a szabványos képnél nézegető segítségével jelenítjük meg:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Több részlet a kép generálásáról

Nézzük meg az `images.generate` paramétereit:

- **model**, a használandó képmodell (például `gpt-image-1`).
- **prompt**, a szöveges utasítás, amely alapján a képet generáljuk. Itt ez: "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, a generált kép mérete (például `1024x1024`, `1536x1024`, `1024x1536` vagy `"auto"`).
- **n**, a generált képek száma. Itt egyet generálunk.

> A képmodellek nem használnak `temperature` paramétert — ez a szöveggenerálás szabályozója. A változatosság növeléséhez hívd meg újra az API-t; a változatosság csökkentéséhez pontosítsd a promptot.

## További képalkotási lehetőségek

Láttad, hogyan lehet néhány sor Python kóddal képet generálni. A `gpt-image` modellek képesek meglévő kép **szerkesztésére** is. Egy képet, opcionális **maszkot** (amely megjelöli a módosítandó területet) és promptot adva megváltoztathatsz egy képrészletet — például kalapot tehetsz a nyuszira.

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# a szerkesztések base64 formátumban is visszaadásra kerülnek
edited_image = base64.b64decode(response.data[0].b64_json)
```

Az alap kép csak a nyuszit tartalmazza; a végleges képen a nyuszin kalap van.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
